In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import files

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs("/content/results", exist_ok=True)

print("TensorFlow:", tf.__version__)

In [ ]:
!wget -q --show-progress "https://archive.ics.uci.edu/static/public/908/realwaste.zip" -O /content/realwaste.zip
!unzip -q /content/realwaste.zip -d /content/realwaste

DATASET_PATH = "/content/realwaste/realwaste-main/RealWaste"

print("Dataset available:", os.path.exists(DATASET_PATH))

In [ ]:
uploaded = files.upload()

In [ ]:
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
class_names = sorted(train_df["label"].unique())
NUM_CLASSES = len(class_names)

class_to_index = {name: index for index, name in enumerate(class_names)}

def prepare_dataframe(df):
    df = df.copy()
    df["filepath"] = df["relative_path"].apply(lambda x: os.path.join(DATASET_PATH, x))
    df["label_index"] = df["label"].map(class_to_index)
    return df

train_df = prepare_dataframe(train_df)
val_df = prepare_dataframe(val_df)
test_df = prepare_dataframe(test_df)

print(class_names)
print("Classes:", NUM_CLASSES)

In [ ]:
def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image, label

def create_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["filepath"].values, df["label_index"].values))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("Datasets ready")

In [ ]:
def get_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1)
    ])

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6
    )
]

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

efficient_base = tf.keras.applications.EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

efficient_base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = get_augmentation()(inputs)
x = efficient_base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

efficient_model = tf.keras.Model(inputs, outputs)

efficient_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

efficient_model.summary()

In [ ]:
efficient_history = efficient_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(efficient_history.history["accuracy"], label="Training Accuracy")
plt.plot(efficient_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNetB0 Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("/content/results/EfficientNetB0_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(efficient_history.history["loss"], label="Training Loss")
plt.plot(efficient_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNetB0 Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig("/content/results/EfficientNetB0_loss.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
efficient_base.trainable = True

for layer in efficient_base.layers[:-30]:
    layer.trainable = False

for layer in efficient_base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

efficient_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

efficient_fine_history = efficient_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

In [ ]:
def evaluate_model(model, model_name):
    test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

    y_true = []
    y_pred = []

    for images, labels in test_ds:
        predictions = model.predict(images, verbose=0)
        predictions = np.argmax(predictions, axis=1)
        y_true.extend(labels.numpy())
        y_pred.extend(predictions)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(model_name, "Test Results")
    print("Accuracy :", f"{accuracy:.4f}")
    print("Precision:", f"{precision:.4f}")
    print("Recall   :", f"{recall:.4f}")
    print("F1 Score :", f"{f1:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    pd.DataFrame(report).transpose().to_csv(
        f"/content/results/{model_name}_classification_report.csv"
    )

    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(figsize=(11, 10))
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        f"/content/results/{model_name}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    results_df = pd.DataFrame([{
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Test Loss": test_loss
    }])

    results_df.to_csv(
        f"/content/results/{model_name}_results.csv",
        index=False
    )

    return results_df

In [ ]:
efficient_results = evaluate_model(efficient_model, "EfficientNetB0")
efficient_model.save("/content/results/EfficientNetB0.keras")
efficient_results

In [ ]:
!zip -r /content/EfficientNetB0_results.zip /content/results
files.download("/content/EfficientNetB0_results.zip")